# Live acceptance through voltsource.AD5780
This notebook is the acceptance record for the live-validation goal. Every DAC operation uses the class exported from `malab.instruments.voltsource`. Tests and assertions are in the cells, not hidden case dispatchers. Supporting diagnostic cases are labeled and cannot substitute for final acceptance. All four outputs must be isolated; only CH1 is connected to the meter. Keep the Â±0.5 mV voltage criterion.

The command-line runner executes these same cells and saves an executed notebook plus a manifest. Default settings prevent hardware access. Measurements and failures are retained in `notebooks/results`. Do not run two instrument notebooks simultaneously.


In [ ]:
from pathlib import Path
import sys, time, math, csv, json
from datetime import datetime, timezone
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"malab/instruments/ad5780.py").exists())
sys.path.insert(0, str(root))
import malab
assert Path(malab.__file__).resolve().parent == root/"malab", "Restart kernel: another malab is imported"
from malab.instruments.voltsource import AD5780
from malab.instruments import Keysight34461A
from malab.instruments.ad5780 import V_PER_BIT, MAX_CODE, voltage_to_code, code_to_voltage
from malab.instruments._line_socket import InstrumentError, CommandError, ProtocolError, InstrumentTimeout
print("Acceptance API:", AD5780, "via malab.instruments.voltsource")
RUN_ID = globals().get("RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
RESULTS = root/"malab/notebooks/results"/RUN_ID
RESULTS.mkdir(parents=True, exist_ok=True)


In [ ]:
RUN_LIVE = False
ALL_OUTPUTS_ISOLATED = False
DAC_ADDRESS = "192.168.1.200"
METER_ADDRESS = "192.168.1.71"
USB_PORT = "COM5"
CHANNEL = 1
VOLTAGE_TOL = 0.0005
RATE_REL_TOL = 0.05
# Optional measured calibration will be configured explicitly in later revisions.
DAC_OPTIONS = dict(address=DAC_ADDRESS, timeout=3)


In [ ]:
def ready():
    assert RUN_LIVE and ALL_OUTPUTS_ISOLATED, "Live work is disabled"
    assert CHANNEL == 1
def save_rows(name, rows):
    path = RESULTS/(name+".csv")
    if rows:
        with path.open("w",newline="") as f:
            writer=csv.DictWriter(f,fieldnames=list(rows[0]))
            writer.writeheader(); writer.writerows(rows)
    print("Saved",len(rows),"measurements:",path)
def connect_dac(timeout=20, **extra):
    deadline=time.monotonic()+timeout
    last=None
    while time.monotonic()<deadline:
        try:
            return AD5780(**(DAC_OPTIONS|extra))
        except (InstrumentError,OSError) as exc:
            last=exc
            time.sleep(0.2)
    raise TimeoutError(f"Fresh connection failed: {last}")



## Diagnostic D01 â€” uncorrected voltage survey (not final acceptance)
Measure raw nominal endpoints and intermediate levels in both directions. Repeat zero to expose drift/hysteresis. A least-squares line is a candidate calibration; independent validation points must pass afterward. This case reports the current Â±0.5 mV failures instead of changing the tolerance.


In [ ]:
ready()
rows=[]
try:
    with connect_dac(use_calibration=False) as dac, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10,nplc=10)
        print(dac.initialize(CHANNEL))
        print("Meter:",meter.get_id())
        for requested in (0,-10,-8,-4,0,4,8,10,0,8,-8,0):
            dac.set_voltage(CHANNEL,requested)
            time.sleep(0.3)
            actual_code=dac.get_code(CHANNEL)
            reported=dac.get_voltage(CHANNEL)
            values=[meter.read_voltage() for _ in range(3)]
            measured=sum(values)/len(values)
            row=dict(requested_V=requested,code=actual_code,reported_V=reported,
                     measured_V=measured,error_V=measured-reported,spread_V=max(values)-min(values))
            rows.append(row); print(row,flush=True)
        dac.set_voltage(CHANNEL,0)
        meter.check_errors()
finally:
    save_rows("raw_voltage_survey",rows)
xs=[r["code"] for r in rows]; ys=[r["measured_V"] for r in rows]
xm=sum(xs)/len(xs); ym=sum(ys)/len(ys)
gain=sum((x-xm)*(y-ym) for x,y in zip(xs,ys))/sum((x-xm)**2 for x in xs)
intercept=ym-gain*xm
residuals=[y-(gain*x+intercept) for x,y in zip(xs,ys)]
candidate=dict(volts_per_code=gain,volts_at_code_zero=intercept,
               zero_code=round(-intercept/gain),min_voltage=intercept,
               max_voltage=intercept+MAX_CODE*gain,
               max_residual=max(map(abs,residuals)),
               nominal_zero_offset=intercept+131072*gain)
(RESULTS/"calibration_candidate.json").write_text(json.dumps(candidate,indent=2)+"\n")
print("Candidate calibration:",candidate)
assert all(abs(r["error_V"])<=VOLTAGE_TOL for r in rows), "Uncorrected voltage survey fails Â±0.5 mV; calibration/diagnosis required"



## Diagnostic D02 â€” ramp timing before repair
The public ramp method starts a 0.2 V sweep at 0.1 V/s. Meter change is evaluated relative to its own initial reading so an offset does not disguise the timing defect. Final absolute accuracy remains a separate required acceptance.


In [ ]:
ready()
rows=[]
try:
    with connect_dac() as dac, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10,nplc=0.2)
        dac.initialize(CHANNEL)
        v0=dac.set_voltage(CHANNEL,0)
        time.sleep(0.2)
        measured0=meter.read_voltage()
        start_code=dac.get_code(CHANNEL)
        target=dac.voltage_to_code(CHANNEL,0.2)
        expected=math.ceil(abs(target-start_code)/10)*math.ceil(10*dac.get_calibration(CHANNEL).volts_per_code/0.1*1e6)/1e6
        t0=time.monotonic()
        accepted=dac.ramp(CHANNEL,0.2,0.1,wait=False)
        while True:
            ta=time.monotonic(); measured=meter.read_voltage(); tb=time.monotonic()
            status=dac.get_status()
            row=dict(elapsed_s=(ta+tb)/2-t0,measured_V=measured,delta_V=measured-measured0,state=status.state)
            rows.append(row)
            assert status.state in ("RUNNING","COMPLETE"), status
            if status.state=="COMPLETE":
                duration=time.monotonic()-t0
                break
            assert time.monotonic()-t0<10,"Ramp did not finish"
        final_code=dac.get_code(CHANNEL)
        dac.set_voltage(CHANNEL,0)
        print(dict(expected_s=expected,actual_s=duration,ratio=duration/expected,final_code=final_code),flush=True)
        assert final_code==target
        assert expected-0.05<=duration<=expected*(1+RATE_REL_TOL)+0.1, "Main API ramp timing fails"
finally:
    save_rows("ramp_timing_baseline",rows)



## V01 — final calibrated SET/GET accuracy
Independent validation points, repeated zero, exact register endpoint and explicit unattainable-voltage rejection. ±0.5 mV stays unchanged.


In [ ]:
ready()
rows=[]
try:
    with connect_dac() as dac, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10,nplc=10)
        dac.initialize(CHANNEL)
        profile=dac.get_calibration(CHANNEL)
        print("Device calibration:",profile)
        assert profile.measured, "CH1 calibration was not loaded from firmware"
        print("Achievable requested range:",dac.get_voltage_limits(CHANNEL))
        for requested in (0, -9, -3.3, -0.05, 0, 0.05, 0.123, 3.3, 9, 10, 0):
            expected=dac.set_voltage(CHANNEL,requested)
            time.sleep(0.2)
            code=dac.get_code(CHANNEL)
            reported=dac.get_voltage(CHANNEL)
            values=[meter.read_voltage() for _ in range(3)]
            measured=sum(values)/len(values)
            row=dict(requested_V=requested,code=code,reported_V=reported,measured_V=measured,
                     error_to_request_V=measured-requested,error_to_get_V=measured-reported)
            rows.append(row); print(row,flush=True)
            assert code==dac.voltage_to_code(CHANNEL,requested), row
            assert reported==expected, row
            assert all(abs(v-requested)<=VOLTAGE_TOL and abs(v-reported)<=VOLTAGE_TOL for v in values), row
        before=dac.get_code(CHANNEL)
        try:
            dac.set_voltage(CHANNEL,-10)
        except ValueError as exc:
            print("Correctly rejected unattainable endpoint:",exc)
        else:
            raise AssertionError("Calibrated -10 V cannot be generated by this hardware")
        assert dac.get_code(CHANNEL)==before
        meter.check_errors()
finally:
    save_rows("calibrated_voltage_acceptance",rows)
print("PASS V01: calibrated SET and GET agree with independent meter within 0.5 mV")
